# Ridge and Lasso Regression

## Ridge Regression

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import Ridge, Lasso, RidgeCV, LassoCV, LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.datasets import load_diabetes, fetch_california_housing
from sklearn.metrics import mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')
np.random.seed(42)
print('All libraries loaded successfully!')

In [ ]:
# ── DATA CREATION ──────────────────────────────────────────────────────────
# The Diabetes dataset has 10 features and 442 samples.
# We split 80% for training and 20% for testing.
# StandardScaler is applied so all features are on the same scale.

diabetes = load_diabetes()
X, y = diabetes.data, diabetes.target
feature_names = diabetes.feature_names

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)  # fit on train, transform train
X_test_s  = scaler.transform(X_test)       # only transform test (no fit)

print(f'Training samples : {X_train.shape[0]}')
print(f'Test samples     : {X_test.shape[0]}')
print(f'Number of features: {X.shape[1]}')
print(f'Features: {feature_names}')

In [ ]:
# ── SOLUTION ───────────────────────────────────────────────────────────
# Fit Ridge regression for each alpha value and record test MSE and coef norm.

alphas = [0.001, 0.01, 0.1, 1, 10, 100, 1000]

test_mse   = []   # store test MSE for each alpha
coef_norms = []   # store L2 norm of coefficients for each alpha

for alpha in alphas:
    # Hint: Create a Ridge model with the current alpha
    # ridge_model = Ridge(alpha=...)
    ridge = Ridge(alpha=alpha)
    
    # Hint: Fit the model on the scaled training data
    ridge.fit(X_train, y_train)
    
    # Hint: Predict on the scaled test data
    # y_pred = ridge_model.predict(...)
    y_predict = ridge.predict(X_test)
    
    # Hint: Compute test MSE using mean_squared_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_predict)
    
    # Hint: Compute L2 norm of coefficients using np.linalg.norm(ridge_model.coef_)
    norm = np.linalg.norm(ridge.coef_)
    
    # Hint: Append mse and norm to the respective lists
    test_mse.append(mse)
    coef_norms.append(norm)

    pass

# Print a summary table
# Hint: Loop through alphas, test_mse, coef_norms and print each row
print(f"{'Alpha':>10}  {'Test MSE':>12}  {'Coef L2 Norm':>14}")
print('-' * 42)
for alpha, mse, norm in zip(alphas, test_mse, coef_norms):
    print(f"{alpha:>10.3f}  {mse:>12.3f}  {norm:>14.3f}")

In [ ]:
# ── VISUALIZATION (Run after your solution above) ───────────────────────────
# This cell visualizes the effect of alpha on test MSE and coefficient norm.
# It also compares Ridge against OLS (no regularization) as a baseline.

# OLS baseline
ols = LinearRegression().fit(X_train_s, y_train)
ols_mse  = mean_squared_error(y_test, ols.predict(X_test_s))
ols_norm = np.linalg.norm(ols.coef_)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Alpha vs Test MSE
axes[0].semilogx(alphas, test_mse, 'o-', lw=2.5, ms=8, color='#E74C3C', label='Ridge')
axes[0].axhline(ols_mse, color='gray', ls='--', lw=2, label=f'OLS (no regularization)')
axes[0].set_xlabel('Alpha (log scale)', fontsize=11, fontweight='bold')
axes[0].set_ylabel('Test MSE', fontsize=11, fontweight='bold')
axes[0].set_title('Effect of Alpha on Test MSE', fontsize=12, fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)
axes[0].annotate('Optimal\nalpha', xy=(alphas[test_mse.index(min(test_mse))], min(test_mse)),
                 xytext=(alphas[test_mse.index(min(test_mse))]*5, min(test_mse)+200),
                 arrowprops=dict(arrowstyle='->', color='black'), fontsize=9)

# Right: Alpha vs Coefficient Norm
axes[1].semilogx(alphas, coef_norms, 'o-', lw=2.5, ms=8, color='#3498DB', label='Ridge')
axes[1].axhline(ols_norm, color='gray', ls='--', lw=2, label='OLS norm')
axes[1].set_xlabel('Alpha (log scale)', fontsize=11, fontweight='bold')
axes[1].set_ylabel('Coefficient L2 Norm', fontsize=11, fontweight='bold')
axes[1].set_title('Effect of Alpha on Coefficient Size', fontsize=12, fontweight='bold')
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.suptitle('Problem 1: Ridge Regression — Effect of Alpha\n'
             'Observation: As alpha increases, coefficients shrink and MSE forms a U-shape',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print('What to observe:')
print('  1. Small alpha  → low shrinkage → coefficients close to OLS → may overfit')
print('  2. Optimal alpha → best test MSE → right balance of bias and variance')
print('  3. Large alpha  → heavy shrinkage → coefficients near zero → underfitting')
print('  4. Ridge NEVER sets any coefficient to exactly zero')

## Lasso

In [ ]:
# ── DATA CREATION ──────────────────────────────────────────────────────────
# Using the same Diabetes dataset from Problem 1.
# X_train_s, X_test_s, y_train, y_test are already created above.

lasso_alphas = [0.001, 0.01, 0.1, 0.5, 1, 5, 10]

print('Lasso alpha values to test:', lasso_alphas)
print(f'Total features in dataset: {X_train_s.shape[1]}')
print('Note: We expect Lasso to select fewer features as alpha increases.')

In [ ]:
# ── SOLUTION ───────────────────────────────────────────────────────────
# Fit Lasso regression for each alpha value.
# Record test MSE, number of non-zero coefficients, and L1 norm.

lasso_mse    = []   # store test MSE
nonzero_coef = []   # store count of non-zero coefficients
l1_norms     = []   # store L1 norm of coefficients

for alpha in lasso_alphas:
    # Hint: Create a Lasso model with the current alpha and max_iter=10000
    lasso_model = Lasso(alpha=alpha, tol=0.0001, selection='cyclic', warm_start=False, positive=False, max_iter=10000)
    
    # Hint: Fit on scaled training data
    lasso_model.fit(X_train_s, y_train)
    
    # Hint: Predict on scaled test data
    y_pred = lasso_model.predict(X_test_s)
    
    # Hint: Compute test MSE
    mse = mean_squared_error(y_test, y_pred)
    lasso_mse.append(mse)

    # Hint: Count non-zero coefficients using np.sum(lasso_model.coef_ != 0)
    nz = np.sum(lasso_model.coef_ != 0)
    nonzero_coef.append(nz)

    # Hint: Compute L1 norm using np.sum(np.abs(lasso_model.coef_))
    l1 = np.sum(np.abs(lasso_model.coef_))
    l1_norms.append(l1)

    pass

# Print summary table
print(f"{'Alpha':>8}  {'Test MSE':>12}  {'Non-zero Coefs':>16}  {'L1 Norm':>10}")
print('-' * 54)
for a, mse, ncef, norm in zip(lasso_alphas, lasso_mse, nonzero_coef, l1_norms):
    print(f"{a:>10}  {mse:>12.4f}  {ncef:>10} {norm:>14.4f}")

## Cross Validation using RidgeCV, LassoCV

### RidgeCV

In [ ]:
# ── YOUR SOLUTION ───────────────────────────────────────────────────────────
# Use RidgeCV and LassoCV to automatically find the best alpha.

# --- Part A: RidgeCV ---
# Hint: ridge_cv = RidgeCV(alphas=alpha_grid, cv=5)
# Hint: ridge_cv.fit(X_train_s, y_train)
# Hint: Access best alpha with ridge_cv.alpha_

ridge_cv = RidgeCV(alphas=alpha_grid, cv=5)
ridge_cv.fit(X_train_s, y_train)

ridge_test_mse = mean_squared_error(y_test, ridge_cv.predict(X_test_s))
ridge_r2       = r2_score(y_test, ridge_cv.predict(X_test_s))

print(f'RidgeCV optimal alpha : {ridge_cv.alpha_:.4f}')
print(f'RidgeCV test MSE      : {ridge_test_mse:.4f}')
print(f'RidgeCV R2 score      : {ridge_r2:.4f}')

### LassoCV

In [ ]:
# --- Part B: LassoCV ---
# Hint: lasso_cv = LassoCV(cv=5, max_iter=10000, random_state=42)
# Hint: lasso_cv.fit(X_train_s, y_train)
# Hint: Access best alpha with lasso_cv.alpha_
# Hint: Count selected features with np.sum(lasso_cv.coef_ != 0)

lasso_cv = LassoCV(cv=5, max_iter=10000, random_state=42)
lasso_cv.fit(X_train_s, y_train)
lasso_cv_nonzero_coef = np.sum(lasso_cv.coef_ != 0)
lasso_cv_alpha = lasso_cv.alpha_

lasso_test_mse = mean_squared_error(y_test, lasso_cv.predict(X_test_s))
lasso_r2       = r2_score(y_test, lasso_cv.predict(X_test_s))
selected       = [feature_names[i] for i in np.where(lasso_cv.coef_ != 0)[0]]

print(f'LassoCV optimal alpha  : {lasso_cv.alpha_:.4f}')
print(f'LassoCV test MSE       : {lasso_test_mse:.4f}')
print(f'LassoCV R2 score       : {lasso_r2:.4f}')
print(f'LassoCV selected features: {selected}')

### Visualization

In [ ]:
# ── VISUALIZATION (Run after your solution above) ───────────────────────────
# This cell shows the cross-validation path for both Ridge and Lasso,
# highlighting the optimal alpha chosen by each method.

# Compute CV scores manually for Ridge (for plotting)
ridge_cv_mse = [-cross_val_score(Ridge(alpha=a), X_train_s, y_train, cv=5,
                 scoring='neg_mean_squared_error').mean() for a in alpha_grid]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: RidgeCV path
axes[0].semilogx(alpha_grid, ridge_cv_mse, lw=2, color='#E74C3C')
axes[0].axvline(ridge_cv.alpha_, color='red', ls='--', lw=2,
                label=f'Optimal alpha = {ridge_cv.alpha_:.4f}')
axes[0].set_xlabel('Alpha (log scale)', fontsize=11, fontweight='bold')
axes[0].set_ylabel('CV Mean Squared Error', fontsize=11, fontweight='bold')
axes[0].set_title('RidgeCV: Cross-Validation Path', fontsize=12, fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# Right: LassoCV path
axes[1].semilogx(lasso_cv.alphas_, lasso_cv.mse_path_.mean(axis=1), lw=2, color='#9B59B6')
axes[1].axvline(lasso_cv.alpha_, color='purple', ls='--', lw=2,
                label=f'Optimal alpha = {lasso_cv.alpha_:.4f}')
axes[1].set_xlabel('Alpha (log scale)', fontsize=11, fontweight='bold')
axes[1].set_ylabel('CV Mean Squared Error', fontsize=11, fontweight='bold')
axes[1].set_title('LassoCV: Cross-Validation Path', fontsize=12, fontweight='bold')
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.suptitle('Problem 3: Optimal Alpha via 5-Fold Cross-Validation\n'
             'The dashed line marks the alpha that minimizes CV error',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print('What to observe:')
print('  1. The CV path has a clear minimum — this is the optimal alpha')
print('  2. Left of minimum: underfitting (too much regularization)')
print('  3. Right of minimum: overfitting (too little regularization)')
print('  4. Always use CV to select alpha — never use the test set')

## Ridge vs Lasso on co-linearity

In [ ]:
# ── DATA CREATION ──────────────────────────────────────────────────────────
# Synthetic multicollinear dataset:
# - 6 features, 200 samples
# - Features 4, 5, 6 are near-copies of features 1, 2, and (1+2)
# - True coefficients: [2.0, -1.5, 0.8, 0.0, 0.0, 0.0]

np.random.seed(42)
n = 200
X_base = np.random.randn(n, 3)
X_mc = np.column_stack([
    X_base[:, 0],                                           # Feature 1
    X_base[:, 1],                                           # Feature 2
    X_base[:, 2],                                           # Feature 3
    X_base[:, 0] + 0.05 * np.random.randn(n),              # Feature 4 ~ Feature 1
    X_base[:, 1] + 0.05 * np.random.randn(n),              # Feature 5 ~ Feature 2
    X_base[:, 0] + X_base[:, 1] + 0.1 * np.random.randn(n) # Feature 6 ~ Feature 1+2
])
true_coefs = np.array([2.0, -1.5, 0.8, 0.0, 0.0, 0.0])
y_mc = X_mc @ true_coefs + 0.5 * np.random.randn(n)

X_mc_tr, X_mc_te, y_mc_tr, y_mc_te = train_test_split(X_mc, y_mc, test_size=0.2, random_state=42)
sc_mc = StandardScaler()
X_mc_tr_s = sc_mc.fit_transform(X_mc_tr)
X_mc_te_s = sc_mc.transform(X_mc_te)

print('Dataset created successfully!')
print(f'Shape: {X_mc.shape}')
print(f'True coefficients: {true_coefs}')

# Show correlation matrix
corr = pd.DataFrame(X_mc, columns=[f'X{i+1}' for i in range(6)]).corr()
print('\nCorrelation between features:')
print(corr.round(2).to_string())

In [ ]:
# ── SOLUTION ───────────────────────────────────────────────────────────
# Fit OLS, Ridge (with RidgeCV), and Lasso (with LassoCV) on the multicollinear dataset.

# --- OLS ---
# Hint: ols_mc = LinearRegression().fit(X_mc_tr_s, y_mc_tr)
ols_mc = LinearRegression().fit(X_mc_tr_s, y_mc_tr)

# --- Ridge with cross-validation ---
# Hint: ridge_mc = RidgeCV(alphas=np.logspace(-3, 4, 50), cv=5).fit(X_mc_tr_s, y_mc_tr)
ridge_mc = RidgeCV(alphas=np.logspace(-3, 4, 50), cv=5).fit(X_mc_tr_s, y_mc_tr)

# --- Lasso with cross-validation ---
# Hint: lasso_mc = LassoCV(cv=5, max_iter=10000, random_state=42).fit(X_mc_tr_s, y_mc_tr)
lasso_mc = LassoCV(cv=5, max_iter=10000, random_state=42).fit(X_mc_tr_s, y_mc_tr)

# --- Print comparison ---
# Hint: Print true_coefs, ols_mc.coef_, ridge_mc.coef_, lasso_mc.coef_
# Hint: Print test MSE for each model
print(f'True coefficients : {true_coefs}')
print(f'OLS coefficients  : {np.round(ols_mc.coef_, 3)}')
print(f'Ridge coefficients: {np.round(ridge_mc.coef_, 3)}')
print(f'Lasso coefficients: {np.round(lasso_mc.coef_, 3)}')

## Ridge vs Lasso on Sparse dataset

In [ ]:
# ── DATA CREATION ──────────────────────────────────────────────────────────
# Synthetic sparse dataset:
# - 50 features, 300 samples
# - Only features 0-4 have non-zero true coefficients
# - Features 5-49 are pure noise

np.random.seed(42)
n_samples, n_features, n_informative = 300, 50, 5

X_sp = np.random.randn(n_samples, n_features)
true_coefs_sp = np.zeros(n_features)
true_coefs_sp[:n_informative] = [3.0, -2.5, 1.8, -1.2, 2.2]  # only first 5 are non-zero

y_sp = X_sp @ true_coefs_sp + 0.5 * np.random.randn(n_samples)

X_sp_tr, X_sp_te, y_sp_tr, y_sp_te = train_test_split(X_sp, y_sp, test_size=0.2, random_state=42)
sc_sp = StandardScaler()
X_sp_tr_s = sc_sp.fit_transform(X_sp_tr)
X_sp_te_s = sc_sp.transform(X_sp_te)

print('Sparse dataset created!')
print(f'Total features: {n_features}')
print(f'Informative features (indices 0-4): {n_informative}')
print(f'True non-zero coefficients: {true_coefs_sp[:5]}')
print(f'Noise features (indices 5-49): {n_features - n_informative}')

In [ ]:
# ── COMPLETE SOLUTION ───────────────────────────────────────────────────────

# OLS — fits all 50 features, overfits noise
ols_sp = LinearRegression().fit(X_sp_tr_s, y_sp_tr)

# Ridge — shrinks all 50 coefficients but keeps all features
ridge_sp = RidgeCV(alphas=np.logspace(-3, 4, 50), cv=5).fit(X_sp_tr_s, y_sp_tr)

# Lasso — selects a sparse subset of features
lasso_sp = LassoCV(cv=5, max_iter=10000, random_state=42).fit(X_sp_tr_s, y_sp_tr)

# Test MSE comparison
print('TEST MSE COMPARISON:')
print(f'OLS   : {mean_squared_error(y_sp_te, ols_sp.predict(X_sp_te_s)):.4f}')
print(f'Ridge : {mean_squared_error(y_sp_te, ridge_sp.predict(X_sp_te_s)):.4f}  (alpha={ridge_sp.alpha_:.4f})')
print(f'Lasso : {mean_squared_error(y_sp_te, lasso_sp.predict(X_sp_te_s)):.4f}  (alpha={lasso_sp.alpha_:.4f})')

# Feature selection by Lasso
selected_indices = np.where(lasso_sp.coef_ != 0)[0]
print(f'\nLasso selected {len(selected_indices)} features out of {n_features}')
print(f'Selected feature indices: {list(selected_indices)}')
print(f'True informative indices: {list(range(n_informative))}')

# Check if Lasso found all informative features
found_all = set(range(n_informative)).issubset(set(selected_indices))
print(f'Lasso found all informative features: {found_all}')

## California Dataset

In [ ]:
# ── DATA CREATION ──────────────────────────────────────────────────────────
from sklearn.metrics import mean_absolute_error

housing = fetch_california_housing()
X_cal, y_cal = housing.data, housing.target
cal_features = housing.feature_names

print('California Housing Dataset')
print(f'  Samples  : {X_cal.shape[0]}')
print(f'  Features : {X_cal.shape[1]}')
print(f'  Feature names: {list(cal_features)}')
print(f'  Target range: {y_cal.min():.2f} to {y_cal.max():.2f} ($100k)')

In [ ]:
# ── COMPLETE SOLUTION ───────────────────────────────────────────────────────

# Step 1: Split the data
X_cal_tr, X_cal_te, y_cal_tr, y_cal_te = train_test_split(
    X_cal, y_cal, test_size=0.2, random_state=42
)

# Step 2: Scale features — fit ONLY on training data to prevent data leakage
sc_cal = StandardScaler()
X_cal_tr_s = sc_cal.fit_transform(X_cal_tr)  # fit + transform on train
X_cal_te_s = sc_cal.transform(X_cal_te)       # only transform on test

# Step 3: Fit models
ols_cal   = LinearRegression().fit(X_cal_tr_s, y_cal_tr)
ridge_cal = RidgeCV(alphas=np.logspace(-3, 4, 50), cv=5).fit(X_cal_tr_s, y_cal_tr)
lasso_cal = LassoCV(cv=5, max_iter=10000, random_state=42).fit(X_cal_tr_s, y_cal_tr)

# Step 4: Build comparison DataFrame
results = pd.DataFrame({
    'Model': ['OLS', 'Ridge', 'Lasso'],
    'Alpha': ['-', f'{ridge_cal.alpha_:.4f}', f'{lasso_cal.alpha_:.4f}'],
    'Train MSE': [
        round(mean_squared_error(y_cal_tr, ols_cal.predict(X_cal_tr_s)), 4),
        round(mean_squared_error(y_cal_tr, ridge_cal.predict(X_cal_tr_s)), 4),
        round(mean_squared_error(y_cal_tr, lasso_cal.predict(X_cal_tr_s)), 4)
    ],
    'Test MSE': [
        round(mean_squared_error(y_cal_te, ols_cal.predict(X_cal_te_s)), 4),
        round(mean_squared_error(y_cal_te, ridge_cal.predict(X_cal_te_s)), 4),
        round(mean_squared_error(y_cal_te, lasso_cal.predict(X_cal_te_s)), 4)
    ],
    'MAE': [
        round(mean_absolute_error(y_cal_te, ols_cal.predict(X_cal_te_s)), 4),
        round(mean_absolute_error(y_cal_te, ridge_cal.predict(X_cal_te_s)), 4),
        round(mean_absolute_error(y_cal_te, lasso_cal.predict(X_cal_te_s)), 4)
    ],
    'R2 Score': [
        round(r2_score(y_cal_te, ols_cal.predict(X_cal_te_s)), 4),
        round(r2_score(y_cal_te, ridge_cal.predict(X_cal_te_s)), 4),
        round(r2_score(y_cal_te, lasso_cal.predict(X_cal_te_s)), 4)
    ]
})

print('CALIFORNIA HOUSING — MODEL COMPARISON:')
print(results.to_string(index=False))

# Step 5: Lasso feature selection
selected = [cal_features[i] for i in np.where(lasso_cal.coef_ != 0)[0]]
dropped  = [cal_features[i] for i in np.where(lasso_cal.coef_ == 0)[0]]
print(f'\nLasso selected features : {selected}')
print(f'Lasso dropped features  : {dropped if dropped else "None (all kept)"}')